# Fine-tuning do saulin — Gemma 4 (E2B / E4B) com QLoRA no Colab (T4)

Projeto CCF726 — SLM para extração de atividades de programações de eventos.

**Como usar:** `Ambiente de execução → Alterar o tipo → T4 GPU`. Ajuste a célula **1. Configuração da rodada** e rode tudo (`Ambiente de execução → Executar tudo`).
Cada rodada grava tudo em `MyDrive/ccf726/<RODADA>/`: configuração, log de treino, adapter LoRA e o GGUF para rodar no PC.

Decisões que este notebook segue (ver `docs/PLANEJAMENTO.md`):
- prompt curto e fixo de `inference/prompt.py`, **o mesmo** usado na inferência;
- template de chat do Gemma 4 **sem thinking**;
- loss só na resposta (`train_on_responses_only`);
- JSON com acentos literais (`ensure_ascii=False`);
- validação = `val` (eventos que não estão no treino); o conjunto de teste **não** é usado aqui.

In [ ]:
# 1. Configuração da rodada — mude aqui e registre em training/EXPERIMENTOS.md
RODADA      = "e2b-r1"                    # nome da rodada (pasta no Drive e nas predições)
MODELO      = "unsloth/gemma-4-E2B-it"    # ou "unsloth/gemma-4-E4B-it"
MAX_SEQ     = 8192                        # teto do contexto; a célula 5 confere os tamanhos reais
LORA_R      = 16
LORA_ALPHA  = 16
LR          = 2e-4
EPOCAS      = 2
BATCH       = 1
GRAD_ACC    = 8                           # batch efetivo = BATCH * GRAD_ACC
WARMUP      = 0.05
SEMENTE     = 3407
AVALIAR_A_CADA = 0.25                     # fração de uma época entre avaliações da loss no val
REPO        = "https://github.com/vitorvmpontes/Projeto-CCF726"
RAMO        = "main"

In [ ]:
# 2. Drive (onde os resultados ficam salvos) e código do repositório
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
SAIDA = f"/content/drive/MyDrive/ccf726/{RODADA}"
os.makedirs(SAIDA, exist_ok=True)

# Repositório privado: crie o segredo GITHUB_TOKEN (ícone de chave, à esquerda) com um token de leitura.
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", RAMO, url, "/content/repo"], check=True)
os.chdir("/content/repo")
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit do repositório:", commit)

In [ ]:
# 3. Dependências (o unsloth fixa torch/transformers/trl/peft compatíveis)
!pip install -q -r training/requirements-colab.txt

In [ ]:
# 4. Dados: conversas de treino/validação no formato do prompt v1
!python -m training.preparar
import json
from datasets import Dataset
def carregar(caminho):
    return [json.loads(l) for l in open(caminho, encoding="utf-8") if l.strip()]
treino = carregar("training/dados/train_chat.jsonl")
val    = carregar("training/dados/val_chat.jsonl")
import csv
ids_rapido = {r["id"] for r in csv.DictReader(open("data/val_eval/manifest_rapido.csv", encoding="utf-8"))}
val_loss = [v for v in val if v["id"] in ids_rapido]   # subconjunto fixo para a loss de validação (mais rápido)
print(len(treino), "treino |", len(val), "val |", len(val_loss), "val p/ loss")
print(treino[0]["messages"][0]["content"][:400])

In [ ]:
# 5. Modelo base em 4 bits + LoRA (só camadas de linguagem)
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = MODELO,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    random_state = SEMENTE,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")   # sem thinking
print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 6. Texto no template de chat + contagem REAL de tokens (decide o max_seq_length)
import numpy as np
tok = getattr(tokenizer, "tokenizer", tokenizer)
def para_texto(exemplos):
    return [tokenizer.apply_chat_template(e["messages"], tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
            for e in exemplos]
def n_tokens(textos):
    return np.array([len(tok(t, add_special_tokens=True)["input_ids"]) for t in textos])

txt_treino, txt_val, txt_val_loss = para_texto(treino), para_texto(val), para_texto(val_loss)
n_tr, n_va = n_tokens(txt_treino), n_tokens(txt_val)
resp = np.array([len(tok(e["messages"][1]["content"], add_special_tokens=False)["input_ids"]) for e in treino])
stats_tokens = {
    "treino": {q: int(np.percentile(n_tr, p)) for q, p in (("p50", 50), ("p95", 95), ("p99", 99), ("max", 100))},
    "val":    {q: int(np.percentile(n_va, p)) for q, p in (("p50", 50), ("p95", 95), ("p99", 99), ("max", 100))},
    "resposta_treino": {q: int(np.percentile(resp, p)) for q, p in (("p50", 50), ("p95", 95), ("max", 100))},
    "tokens_treino_total": int(n_tr.sum()),
}
print(json.dumps(stats_tokens, indent=1))
longos = int((n_tr > MAX_SEQ).sum())
print(f"exemplos de treino acima de {MAX_SEQ} tokens: {longos}")
# Nunca truncar: exemplo cortado ensinaria JSON incompleto. Os que passam do teto ficam de fora (e são registrados).
manter = n_tr <= MAX_SEQ
ds_treino = Dataset.from_dict({"text": [t for t, m in zip(txt_treino, manter) if m]})
ds_val    = Dataset.from_dict({"text": [t for t, n in zip(txt_val_loss, n_tokens(txt_val_loss)) if n <= MAX_SEQ]})
print(len(ds_treino), "exemplos de treino |", len(ds_val), "de validação (loss)")
print(txt_treino[0][:300], "...", txt_treino[0][-200:])

In [ ]:
# 7. Treinador: loss só na resposta do modelo
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import math

passos_epoca = math.ceil(len(ds_treino) / (BATCH * GRAD_ACC))
avaliar = max(10, int(passos_epoca * AVALIAR_A_CADA))
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_treino,
    eval_dataset = ds_val,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ,
        per_device_train_batch_size = BATCH,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = GRAD_ACC,
        num_train_epochs = EPOCAS,
        warmup_ratio = WARMUP,
        learning_rate = LR,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = avaliar,
        save_strategy = "steps",
        save_steps = avaliar,
        save_total_limit = 2,
        load_best_model_at_end = True,          # fica com o checkpoint de menor loss no val
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        seed = SEMENTE,
        output_dir = "/content/checkpoints",
        report_to = "none",
    ),
)

# Marcadores de turno do template (Gemma 4: "<|turn>user\n" / "<|turn>model\n"), detectados do próprio template
amostra = tokenizer.apply_chat_template([{"role": "user", "content": "AAA"}, {"role": "assistant", "content": "BBB"}],
                                        tokenize=False).removeprefix("<bos>")
inst = amostra[:amostra.index("AAA")]
inst = inst[inst.rfind("<"):] if "<" in inst else inst      # só o marcador do turno do usuário
resp_part = amostra[amostra.index("AAA") + 3: amostra.index("BBB")].lstrip()
resp_part = resp_part[resp_part.rfind("<"):] if "<" in resp_part else resp_part
print("instruction_part:", repr(inst), "| response_part:", repr(resp_part))
trainer = train_on_responses_only(trainer, instruction_part = inst, response_part = resp_part)

# Conferência: os rótulos visíveis devem começar no JSON da resposta
lab = trainer.train_dataset[0]["labels"]
visivel = tok.decode([t for t in lab if t != -100])
print("loss calculada sobre:", visivel[:150], "...")
assert visivel.lstrip().startswith('{"atividades"'), "máscara errada: a loss não começa na resposta"
print(f"{passos_epoca} passos/época · avaliação a cada {avaliar} passos")

In [ ]:
# 8. Treino
import time
t0 = time.time()
stats = trainer.train()
duracao_min = (time.time() - t0) / 60
mem_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"{duracao_min:.1f} min · pico de memória {mem_gb:.1f} GB · melhor checkpoint: {trainer.state.best_model_checkpoint}")

import pandas as pd
log = pd.DataFrame(trainer.state.log_history)
log.to_csv(f"{SAIDA}/log_treino.csv", index=False)
ax = log.dropna(subset=["loss"]).plot(x="step", y="loss", figsize=(7, 3), label="treino")
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="validação", marker="o")
ax.figure.savefig(f"{SAIDA}/curva_loss.png", dpi=150, bbox_inches="tight")

In [ ]:
# 9. Teste rápido: o modelo gera JSON válido em 3 exemplos de validação?
FastModel.for_inference(model)
from inference.postprocess import pos_processar
for e in val_loss[:3]:
    entrada = tokenizer.apply_chat_template(e["messages"][:1], tokenize=False, add_generation_prompt=True)
    ids = tok(entrada, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=4096, do_sample=False)
    saida = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    alvo = json.loads(e["messages"][1]["content"])["atividades"]
    pp = pos_processar(saida)
    print(f"{e['id']}: {len(pp.atividades)} atividades geradas / {len(alvo)} no rótulo | aviso: {pp.erro or 'ok'}")
    print("   ", saida[:200].replace("\n", " "), "...")

In [ ]:
# 10. Salvar: configuração da rodada, adapter LoRA e GGUF (Q4_K_M) no Drive
cfg = {
    "rodada": RODADA, "modelo_base": MODELO, "commit_repo": commit, "max_seq": MAX_SEQ,
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "epocas": EPOCAS, "batch_efetivo": BATCH * GRAD_ACC,
    "warmup": WARMUP, "semente": SEMENTE, "exemplos_treino": len(ds_treino), "removidos_por_tamanho": longos,
    "tokens": stats_tokens, "duracao_min": round(duracao_min, 1), "pico_memoria_gb": round(mem_gb, 1),
    "gpu": torch.cuda.get_device_name(0), "melhor_checkpoint": trainer.state.best_model_checkpoint,
    "melhor_eval_loss": trainer.state.best_metric, "train_loss_final": stats.training_loss,
}
json.dump(cfg, open(f"{SAIDA}/rodada.json", "w"), ensure_ascii=False, indent=2)
model.save_pretrained(f"{SAIDA}/adapter")
tokenizer.save_pretrained(f"{SAIDA}/adapter")
model.save_pretrained_gguf(f"/content/gguf_{RODADA}", tokenizer, quantization_method = "q4_k_m")
!cp /content/gguf_{RODADA}/*.gguf "{SAIDA}/" && ls -lh "{SAIDA}"
print(json.dumps(cfg, ensure_ascii=False, indent=1))

## Depois do treino (no seu PC, com a RTX 3050)

1. Baixe do Drive o `.gguf` desta rodada (`MyDrive/ccf726/<RODADA>/`) para `models/` no repositório (pasta ignorada pelo git).
2. Suba o servidor (llama.cpp, binário do Windows com CUDA):
   `llama-server -m models\<arquivo>.gguf --jinja -c 12288 -ngl 99 --port 8080`
3. Gere as predições da validação rápida e avalie:
   ```
   python -m evaluation.run_slm --nome <RODADA> --manifest data/val_eval/manifest_rapido.csv
   python -m evaluation.evaluate --pred-dir evaluation/predictions/val/<RODADA> --manifest data/val_eval/manifest_rapido.csv --nome val/<RODADA> --pos-processar
   ```
4. Registre a rodada em `training/EXPERIMENTOS.md` (copie números de `rodada.json` e do `evaluate`).